# Bắt đầu: phân tích chuyến bay

Dữ liệu thật BTS tháng01/2025.

**Câu hỏi:** Hãng/sân bay nào có tỷ lệ đến trễ cao trong tháng này?

**Dự đoán trước:** mỗi dòng kết quả là một chuyến hay một nhóm?

In [ ]:
from pathlib import Path
import sys
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'flight_lab').is_dir())
sys.path.insert(0,str(root))
from flight_lab.learning import database,read_sql
db=database()
print('Database chỉ đọc:',db)


### 1. Đọc nguồn đã nhập
Một dòng flights là một bản ghi chuyến, không phải hành khách.

In [ ]:
read_sql('SELECT flight_date,carrier_id,origin_id,dest_id,crs_dep_time,arr_delay,cancelled,diverted FROM flights LIMIT 10')

### 2. Kiểm tra chất lượng
Không thay chuyến hủy/chuyển hướng hoặc NULL thành đến đúng giờ.

In [ ]:
from flight_lab.analysis import quality,queries,reconcile
quality(db)

### 3. JOIN đúng cấp
Cùng bảng airports nối hai lần. LIMIT chỉ để đọc vài dòng, không phải chọn mẫu thống kê.

In [ ]:
read_sql('SELECT f.flight_date,c.reporting_code,o.code origin,d.code destination,f.arr_delay FROM flights f JOIN carriers c USING(carrier_id) JOIN airports o ON o.airport_id=f.origin_id JOIN airports d ON d.airport_id=f.dest_id LIMIT 10')

### 4. SQL: phân loại trước, tổng hợp sau
Trễ từ15phút; mẫu số eligible. NULLIF chặn chia0.

In [ ]:
read_sql("SELECT c.reporting_code,COUNT(*) total,SUM(eligible) eligible,SUM(status='delayed') delayed,100.0*SUM(status='delayed')/NULLIF(SUM(eligible),0) delay_rate_pct FROM flight_status f JOIN carriers c USING(carrier_id) GROUP BY c.carrier_id,c.reporting_code ORDER BY delay_rate_pct DESC")

### 5. pandas đọc lại nguồn để tính độc lập
Giữ ZIP/CSV ban đầu. Việc đối chiếu đọc theo khối50.000dòng; có thể mất vài chục giây.

In [ ]:
tables,checks,plot=reconcile(db)
assert tables['airline_summary'].total.sum()==tables['daily_summary'].total.sum()
tables['airline_summary']

### 6. So ngày thường/cuối tuần
Phân biệt tỷ lệ theo chuyến với số chuyến/ngày; mẫu số ngày là ngày lịch.

In [ ]:
tables['day_type_summary']

### 7. Vẽ bốn biểu đồ
Xuất sau khi SQL/pandas khớp. Heatmap là giờ khởi hành dự kiến địa phương, không phải giờ đến.

In [ ]:
from flight_lab.analyze import analyze
out=analyze(db,out=root/'outputs/notebook')
from IPython.display import Image,display
for path in sorted(out.glob('*.png')):display(Image(filename=str(path)))

### 8. Đọc ba nhận xét và giới hạn
Không đánh giá chất lượng tổng thể hoặc nguyên nhân chỉ từ một tháng.

In [ ]:
print((out/'report.md').read_text(encoding='utf-8'))

## Tự làm

Chọn một hãng, tự tính tỷ lệ bằng delayed/eligible. Thử khoảng01–15/01và so với cả tháng. Viết3nhận xét có mẫu số, kỳ và giới hạn.

Xem lời giải riêng sau khi thử. Không ghi vào database thật.